# B2-020 — Session 4: Fine-Tune a Language Transformer

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).



## 1. Checkpoint and state boundary

The student-facing `tiny_encoder_state.py` contains only architecture, trained encoder tensors, `ENCODER_STATE_HASH`, and the public `canonical_encoder_state_hash` verifier. It has no objectives, losses, probes, targets, or trace. Before loading, recompute `canonical_encoder_state_hash(ENCODER_TENSORS)` and require it to equal `ENCODER_STATE_HASH` and the expected literal; comparing a declared hash to itself does not detect a tensor edit. Do not load the author-only checkpoint.

**Checkpoint 1A.** Which module may p20/p21 load?

**Checkpoint 1B.** Which fields are deliberately absent?

In [ ]:
import torch
torch.manual_seed(20260812)
ATOL = 1e-6
RTOL = 1e-6
assert torch.get_num_threads() >= 1

## 2. Attach a task head

For sequence classification, select or mean-pool encoder states to `(B,8)`, then apply a new `Linear(8,C)` head. The head begins from a seeded random state even though the encoder is trained.

**Checkpoint 2A.** Trace logits for `C=2`.

**Checkpoint 2B.** Why is the head not pretrained?

## 3. Freeze parameters explicitly

A frozen stage sets every encoder parameter's `requires_grad=False` and gives the optimizer only classifier parameters. Freezing is a parameter-and-optimizer contract: merely omitting `backward()` or using `eval()` does not freeze learning.

**Checkpoint 3A.** Audit trainable names.

**Checkpoint 3B.** Why must optimizer membership also be checked?

## 4. Worked frozen classification step

Load the exact encoder state, snapshot encoder and head tensors, compute one supervised classification loss, backpropagate, step the head-only optimizer, and compare snapshots. The head must change and the encoder must remain tensor-equal.

**Checkpoint 4A.** Which snapshot is the negative control?

**Checkpoint 4B.** What does a changed frozen tensor prove?

## 5. Unfreeze with a deliberate optimizer

Set encoder gradients back on and create a new optimizer containing encoder plus head, usually with a smaller encoder learning rate. The required practice pins groups explicitly so stale head-only membership cannot survive the transition.

**Checkpoint 5A.** Which group is added?

**Checkpoint 5B.** Why is a new optimizer defensible here?

## 6. Held-out evaluation

Use disjoint train, validation, and test split IDs from the literal fixture. Select hyperparameters with validation only, perform one final test evaluation, and report accuracy plus the fixed split audit.

**Checkpoint 6A.** Which split may choose an epoch?

**Checkpoint 6B.** Why is a training row in evaluation leakage?

## 7. Common pitfalls

Failures include loading random weights, loading the author-only checkpoint, trusting a self-declared hash without recomputing it from `ENCODER_TENSORS`, freezing gradients but optimizing stale copies, evaluating on train rows, and declaring success from training loss. Repair with canonical hash, identity, snapshot, and split assertions.

**Checkpoint 7A.** Which check rejects a tensor edit that leaves its declared hash unchanged?

**Checkpoint 7B.** Which check rejects a frozen-head failure?

## 8. Exam connections and Going deeper

Expect parameter-change audits, freeze/unfreeze protocol choices, head/loss pairing, and held-out metric reasoning. Going deeper names discriminative learning rates and gradual unfreezing as optional future techniques.

**Checkpoint 8A.** Which practices load the state?

**Checkpoint 8B.** Why is the test split used once?

## Collected checkpoint answers

**Answer 1A.** only `tiny_encoder_state.py` may be loaded.  **Answer 1B.** losses, probes, targets, and trace are absent.

**Answer 2A.** `(B,8) -> (B,2)`.  **Answer 2B.** the classifier begins seeded but untrained.

**Answer 3A.** encoder names have `requires_grad=False`.  **Answer 3B.** the optimizer must omit them too.

**Fully worked computation.** Here is a literal two-coordinate slice of one pooled `(B,8)` row: `h = [0.40, -0.20]`.  Let `head_before = [[0.50, -0.10], [-0.20, 0.40]]` with zero bias and label 1; then `logits = [0.22, -0.16]`, `softmax([0.22, -0.16]) = [0.5939, 0.4061]`, and `loss = -log(0.4061) = 0.9011`.  The logit gradient is `[0.5939, -0.5939]`, so `dL/dW = [[0.2376, -0.1188], [-0.2376, 0.1188]]`.  With learning rate 0.1, `head_after = [[0.4762, -0.0881], [-0.1762, 0.3881]]`.  In the frozen stage, `encoder_before = [0.40, -0.20]` and `encoder_after = [0.40, -0.20]`: the optimizer contains only `head.parameters()`, so the numeric head update is permitted while the encoder snapshot is unchanged.

**Answer 4A.** the encoder snapshot is the negative control.  **Answer 4B.** a changed frozen tensor proves the protocol failed.

**Answer 5A.** encoder parameters join the second optimizer.  **Answer 5B.** membership changes deliberately at unfreeze.

**Answer 6A.** validation chooses an epoch.  **Answer 6B.** train rows in evaluation leak information.

**Answer 7A.** recompute `canonical_encoder_state_hash(ENCODER_TENSORS)` and compare it with both the declared and expected hashes.  **Answer 7B.** optimizer membership and snapshots.

**Answer 8A.** p20 and p21.  **Answer 8B.** the test split is a one-time final estimate.